In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import pennylane as qml
from pennylane.qnn import TorchLayer

# ══════════════════════════════════════════════════════════════════════════════
# HybridResNet v3 + QSLP Adversarial Training (Complete, Fixed File)
#
# ARCHITECTURE FLOW (same as v3, nothing removed):
#   Input (B,1,32,32)
#   → stem (Conv 1→16)
#   → stage1 (ResBlock 16→16 ×2)
#   → stage2 (ResBlock 16→32 ×2, stride=2)
#   → stage3 (ResBlock 32→64 ×2, stride=2)
#   → GAP → (B, 64)            ← QNI-CCP hook reads HERE
#   → QuantumBridge (64→16)    ← compress to n_qubits*2
#   → QuantumLayer  (16→24)    ← PennyLane circuit, 3-axis measurement
#   → Classifier    (24→10)    ← final logits
#
# ADVERSARIAL TRAINING (QSLP from paper Algorithm 1):
#   loss = 0.50 * loss_clean        (unperturbed images)
#        + 0.15 * loss_qni_ccp      (latent-space perturbation via hook)
#        + 0.10 * loss_fgsm         (pixel-space, single-step)
#        + 0.15 * loss_pgd          (pixel-space, iterative, 7 steps)
#        + 0.10 * centroid_reg      (pulls features toward class mean)
#
# WHY FeatureHook EXISTS:
#   The forward pass is a single chain with no built-in way to read
#   intermediate activations. QNI-CCP needs the 64-dim GAP output
#   BEFORE the bridge. A PyTorch forward hook intercepts and stores
#   the GAP output mid-forward-pass without modifying the model.
#   It is registered, used, and immediately removed — no memory leak.
# ══════════════════════════════════════════════════════════════════════════════


# ─────────────────────────────────────────────
# SEEDING
# ─────────────────────────────────────────────
def seed_all(seed=42):
    torch.manual_seed(seed)        # reproducible PyTorch ops
    np.random.seed(seed)           # reproducible NumPy ops
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
n_qubits     = 8
q_depth      = 6
q_out_dim    = 3 * n_qubits    # 24 — three Pauli measurements per qubit
batch_size   = 32
num_classes  = 10

# ── Adversarial fine-tuning config ──
num_epochs   = 40              # fewer epochs than clean training (fine-tuning)
lr           = 0.0002          # lower than clean training (0.0005) to avoid destroying pretrained weights
weight_decay = 3e-4

# ── Attack hyperparameters (from paper Table 3) ──
FGSM_EPS   = 0.03              # ε for FGSM: maximum single-step perturbation
PGD_EPS    = 0.10              # ε for PGD: maximum total perturbation (ℓ∞ ball)
PGD_ALPHA  = 0.01              # α for PGD: step size per iteration
PGD_ITERS  = 7                 # k for PGD: number of iterative steps
EPSILON_Q  = 1.0               # QNI-CCP scaling factor (α=β=1.0 from paper)

# ── Loss weights (from paper Algorithm 1, sum = 1.00) ──
W_CLEAN    = 0.50              # dominant: preserve clean accuracy
W_QNI      = 0.15              # QNI-CCP: latent-space robustness
W_FGSM     = 0.10              # FGSM: lightweight pixel-space defence
W_PGD      = 0.15              # PGD: stronger pixel-space defence
W_CENTROID = 0.10              # centroid reg: compact, stable feature clusters


# ─────────────────────────────────────────────
# TRANSFORMS — identical to v3 clean training
# ─────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))   # maps [0,1] → [-1,1]
])
eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


# ─────────────────────────────────────────────
# DATASETS & LOADERS
# ─────────────────────────────────────────────
TRAIN_PATH = 'virus_MNIST dataset/train_balanced_v2'
VAL_PATH   = 'virus_MNIST dataset/val'
TEST_PATH  = 'virus_MNIST dataset/test'

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    val_dataset   = ImageFolder(VAL_PATH,   transform=eval_transform)
    test_dataset  = ImageFolder(TEST_PATH,  transform=eval_transform)
    print(f"Datasets loaded: {len(train_dataset)} train | "
          f"{len(val_dataset)} val | {len(test_dataset)} test")
except Exception as e:
    print(f"Error loading datasets: {e}")
    raise

try:
    labels = [label for _, label in train_dataset.samples]
    class_weights = compute_class_weight(
        class_weight='balanced', classes=np.unique(labels), y=labels
    )
    class_weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    print("Class weights computed:", np.round(class_weights, 3))
except Exception as e:
    print(f"Could not compute class weights: {e}. Using uniform.")
    class_weight_tensor = torch.ones(num_classes).to(device)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)


# ══════════════════════════════════════════════════════════════════════════════
# FULL MODEL DEFINITION — HybridResNet v3
# (identical architecture to clean training file)
# ══════════════════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────
# SAM — Sharpness-Aware Minimisation
# ─────────────────────────────────────────────
# NOTE: SAM is used only for clean training, not for adversarial fine-tuning.
# Adversarial training itself acts as a strong regulariser, so standard AdamW
# is sufficient and faster for the fine-tuning phase.
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer_cls, rho=0.05, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer_cls(self.param_groups, **kwargs)
        self.param_groups   = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                e_w = p.grad * scale.to(p)
                p.add_(e_w)
                self.state[p]["e_w"] = e_w
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.sub_(self.state[p]["e_w"])
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    def _grad_norm(self):
        norms = [
            p.grad.norm(p=2).to(self.param_groups[0]["params"][0])
            for group in self.param_groups
            for p in group["params"]
            if p.grad is not None
        ]
        return torch.stack(norms).norm(p=2)

    def load_state_dict(self, state_dict):
        super().load_state_dict(state_dict)
        self.base_optimizer.param_groups = self.param_groups


# ─────────────────────────────────────────────
# FOCAL LOSS — identical to v3
# ─────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss with class weighting and label smoothing.
    Data flow: logits (B,10) → per-sample CE → focal modulation → mean scalar
    """
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.15):
        super().__init__()
        self.weight          = weight
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(
            inputs, targets,
            weight          = self.weight,
            label_smoothing = self.label_smoothing,
            reduction       = 'none'
        )
        pt         = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


# ─────────────────────────────────────────────
# QUANTUM CIRCUIT — identical to v3
# ─────────────────────────────────────────────
# 8 qubits, 6 variational layers, 3-axis Pauli measurement → 24 features
# Data re-uploading: input angles fused with trainable weights at each layer
# Brick-layer CRZ entanglement alternates even/odd pairs per layer

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # Initial angle encoding: RY then RZ per qubit
    # inputs shape: (B, 2*n_qubits) = (B, 16)
    for i in range(n_qubits):
        qml.RY(inputs[..., i],            wires=i)  # first 8 angles → RY
        qml.RZ(inputs[..., i + n_qubits], wires=i)  # next  8 angles → RZ

    # Variational layers with data re-uploading + brick CRZ entanglement
    for l in range(weights.shape[0]):
        # Brick-layer entanglement: alternates even/odd pairs each layer
        if l % 2 == 0:
            for i in range(0, n_qubits - 1, 2):
                qml.CRZ(weights[l, i, 2], wires=[i, i + 1])   # even pairs
        else:
            for i in range(1, n_qubits - 1, 2):
                qml.CRZ(weights[l, i, 2], wires=[i, i + 1])   # odd pairs

        # Re-upload: fuse data angles with trainable weights per layer
        for i in range(n_qubits):
            qml.RY(weights[l, i, 0] + inputs[..., i],            wires=i)
            qml.RZ(weights[l, i, 1] + inputs[..., i + n_qubits], wires=i)

    # 3-axis Pauli measurement: Z, X, Y per qubit → 24 expectation values
    measurements = []
    for i in range(n_qubits):
        measurements.append(qml.expval(qml.PauliZ(i)))  # Z-axis
        measurements.append(qml.expval(qml.PauliX(i)))  # X-axis
        measurements.append(qml.expval(qml.PauliY(i)))  # Y-axis
    return measurements

weight_shapes = {"weights": (q_depth, n_qubits, 3)}


# ─────────────────────────────────────────────
# BUILDING BLOCKS — identical to v3
# ─────────────────────────────────────────────
class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation: global avg pool → FC → sigmoid → channel rescale.
    Data flow: (B,C,H,W) → pool → (B,C) → FC → (B,C) → sigmoid → rescale → (B,C,H,W)
    Lets the network learn which channels are most important per sample.
    """
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 4), bias=False),
            nn.ReLU(),
            nn.Linear(max(channels // reduction, 4), channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        scale = self.pool(x).view(b, c)           # (B, C)
        scale = self.fc(scale).view(b, c, 1, 1)   # (B, C, 1, 1)
        return x * scale                           # channel-wise rescaling


class ResBlock(nn.Module):
    """
    Residual block with SE attention and stochastic depth.
    Data flow: x → [conv3→BN→ReLU→Dropout→conv3→BN] → SE → stoch_depth → + skip → ReLU
    Skip connection: 1×1 conv if shape changes, else identity.
    """
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.20, drop_path=0.10):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        self.se             = SEBlock(out_ch)
        self.drop_path_rate = drop_path
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if (stride != 1 or in_ch != out_ch) else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.conv_block(x)
        out = self.se(out)
        # Stochastic depth: randomly drop entire block during training
        if self.training and self.drop_path_rate > 0:
            keep_prob     = 1 - self.drop_path_rate
            random_tensor = (
                torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep_prob
            ).float()
            out = out * random_tensor / keep_prob   # scale to maintain expectation
        return self.relu(out + self.skip(x))


class QuantumBridge(nn.Module):
    """
    Compresses 64-dim backbone features → 2*n_qubits angles for the quantum circuit.
    Data flow: (B,64) → Linear(64,32) → LayerNorm → GELU → Dropout → Linear(32,16)
               → sigmoid scaling → (B,16) angles in [0, π] range
    The angle_scale and angle_bias are learnable — model decides the angle range.
    """
    def __init__(self, in_features, n_qubits):
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(in_features, 32),     # 64 → 32
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(0.35),              # strong dropout at quantum interface
            nn.Linear(32, n_qubits * 2)   # 32 → 16 (= 2 angles per qubit)
        )
        # Learnable scale and bias to map outputs into valid quantum angle range
        self.angle_scale = nn.Parameter(torch.ones(n_qubits * 2) * torch.pi)
        self.angle_bias  = nn.Parameter(torch.zeros(n_qubits * 2))

    def forward(self, x):
        x = self.project(x)                             # (B, 16)
        return self.angle_scale * torch.sigmoid(x) + self.angle_bias  # (B, 16) in angle space


# ─────────────────────────────────────────────
# MAIN MODEL — HybridResNet v3
# ─────────────────────────────────────────────
class HybridResNet(nn.Module):
    """
    Full hybrid quantum-classical ResNet.

    Backbone channels: 16 → 32 → 64 (reduced from v2's 32→64→128 to cut overfitting)
    Quantum circuit: 8 qubits × 6 layers → 24 measurement features

    Complete data flow:
      (B,1,32,32)
      → stem:       Conv(1,16) → BN → ReLU               → (B,16,32,32)
      → stage1:     ResBlock(16,16) × 2                   → (B,16,32,32)
      → stage2:     ResBlock(16,32,stride=2) + ResBlock   → (B,32,16,16)
      → stage3:     ResBlock(32,64,stride=2) + ResBlock   → (B,64, 8, 8)
      → GAP:        AdaptiveAvgPool2d(1) → flatten        → (B,64)
      → bridge:     QuantumBridge(64→16)                  → (B,16)
      → q_layer:    PennyLane quantum circuit             → (B,24)
      → classifier: Linear(24,48) → ReLU → Dropout → Linear(48,10) → (B,10)
    """
    def __init__(self, n_qubits, q_out_dim, num_classes, dropout=0.35):
        super().__init__()

        # ── Classical backbone (v3 channel sizes) ─────────────────
        self.stem = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)
        )
        # Progressive stochastic depth: deeper stages drop more aggressively
        self.stage1 = nn.Sequential(
            ResBlock(16, 16, drop_path=0.10),
            ResBlock(16, 16, drop_path=0.10)
        )
        self.stage2 = nn.Sequential(
            ResBlock(16, 32, stride=2, drop_path=0.15),
            ResBlock(32, 32,           drop_path=0.15)
        )
        self.stage3 = nn.Sequential(
            ResBlock(32, 64, stride=2, drop_path=0.20),
            ResBlock(64, 64,           drop_path=0.20)
        )
        # GAP: reduces (B,64,H,W) → (B,64,1,1) regardless of spatial size
        # QNI-CCP HOOK IS ATTACHED HERE — GAP output = latent space for QNI-CCP
        self.gap = nn.AdaptiveAvgPool2d(1)

        # ── Quantum bridge + circuit ───────────────────────────────
        self.bridge  = QuantumBridge(in_features=64, n_qubits=n_qubits)
        self.q_layer = TorchLayer(quantum_circuit, weight_shapes)

        # ── Classifier head ───────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(q_out_dim, q_out_dim * 2),    # 24 → 48
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(q_out_dim * 2, num_classes)   # 48 → 10
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)                    # (B, 16, 32, 32)
        x = self.stage1(x)                  # (B, 16, 32, 32)
        x = self.stage2(x)                  # (B, 32, 16, 16)
        x = self.stage3(x)                  # (B, 64,  8,  8)
        x = self.gap(x)                     # (B, 64,  1,  1)
        x = x.view(x.size(0), -1)           # (B, 64)   ← QNI-CCP hook reads here
        x = self.bridge(x)                  # (B, 16)
        x = self.q_layer(x)                 # (B, 24)
        return self.classifier(x)           # (B, 10)


# ══════════════════════════════════════════════════════════════════════════════
# QSLP ADVERSARIAL COMPONENTS
# ══════════════════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────
# FEATURE HOOK — why it's necessary
# ─────────────────────────────────────────────
# QNI-CCP needs the 64-dim GAP output (after GAP, before bridge).
# The forward() method doesn't return it separately.
# A PyTorch hook intercepts the GAP layer's output mid-forward-pass.
# Without this, we'd have to split the forward() into two functions
# and rewrite the model — the hook is cleaner and non-invasive.
#
# HOW IT WORKS:
#   hook = FeatureHook(model.gap)   → register callback on GAP
#   _ = model(x)                    → forward runs; GAP fires; hook stores output
#   z = hook.features               → (B, 64) GAP output
#   hook.remove()                   → deregister to free memory
#
# The hook stores output BEFORE the view/flatten in forward(),
# so we get shape (B, 64, 1, 1) and flatten it ourselves.

class FeatureHook:
    """Intercepts and stores the output of a single layer during forward pass."""
    def __init__(self, layer):
        self.features = None
        # register_forward_hook: called automatically after layer.forward()
        self._handle  = layer.register_forward_hook(self._capture)

    def _capture(self, module, input, output):
        # output from GAP: (B, 64, 1, 1) — store it
        self.features = output

    def remove(self):
        self._handle.remove()   # always call this after use to prevent memory leaks


def get_gap_features(model, x):
    """
    Runs a full forward pass and returns the flattened GAP output (B, 64).
    This is the latent space where QNI-CCP and centroid reg operate.

    Data flow:
      x (B,1,32,32) → full model forward → GAP fires → hook stores (B,64,1,1)
      → flatten → (B, 64)

    Why not just split model.forward():
      Splitting would require rewriting the model class.
      The hook approach is cleaner and keeps the model untouched.
    """
    hook = FeatureHook(model.gap)
    with torch.no_grad():
        _ = model(x)                                          # trigger forward; hook fires
    z = hook.features.view(hook.features.size(0), -1)        # (B,64,1,1) → (B,64)
    hook.remove()
    return z


# ─────────────────────────────────────────────
# CLASS CENTROID COMPUTATION
# ─────────────────────────────────────────────
# Centroid of class c = mean of all (B,64) feature vectors where label==c.
# μ_c = (1/N_c) * Σ z_i   for all i where y_i == c
#
# Used by QNI-CCP (target: push features toward wrong centroid)
# and centroid regularisation (target: pull features toward correct centroid).
# Recomputed every 5 epochs as the model updates shift the feature space.

def compute_class_centroids(model, dataloader, device, num_classes):
    """
    Iterates the full training set to compute mean feature vector per class.

    Returns: centroids (num_classes, 64) — one mean vector per class
    Data flow: batches → get_gap_features → accumulate per class → divide by count
    """
    model.eval()
    sum_features = torch.zeros(num_classes, 64, device=device)  # accumulator
    count        = torch.zeros(num_classes,     device=device)  # per-class sample count

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="Computing centroids", leave=False):
            x, y  = x.to(device), y.to(device)
            feats = get_gap_features(model, x)      # (B, 64)
            for c in range(num_classes):
                mask = (y == c)                     # which samples belong to class c
                if mask.sum() > 0:
                    sum_features[c] += feats[mask].sum(dim=0)
                    count[c]        += mask.sum()

    count     = count.clamp(min=1.0)                # avoid division by zero
    centroids = sum_features / count.unsqueeze(1)   # (num_classes, 64)
    return centroids


# ─────────────────────────────────────────────
# QNI-CCP — Quantum Noise Injection Class Center Perturbation
# ─────────────────────────────────────────────
# Operates in LATENT SPACE (64-dim GAP output), not pixel space.
# This is the key contribution of the paper: pixel attacks are diluted
# by downsampling, but latent attacks reach the quantum layer directly.
#
# STEPS:
#   1. Extract z (B,64) via hook + run bridge+q_layer+classifier to get logits
#   2. Compute sensitivity S = ∂L/∂z — which features most affect loss
#   3. Pick random wrong class c' ≠ y for each sample
#   4. Perturb: z' = z + ε_q * (S_norm ⊙ (μ_{c'} - z))
#      → pushes features toward wrong-class centroid, weighted by sensitivity
#   5. Return perturbed logits so we can compute CE loss on them

def qni_ccp_loss(model, x, y, centroids, epsilon_q=1.0):
    """
    Computes the QNI-CCP adversarial loss in latent space.

    Args:
        model     : HybridResNet v3 (needs gap, bridge, q_layer, classifier)
        x         : input batch (B,1,32,32)
        y         : true labels (B,)
        centroids : (num_classes, 64) class mean features
        epsilon_q : perturbation scale (paper: α=β=1.0)

    Returns:
        loss_qni : scalar CE loss on perturbed latent features

    Data flow:
        x → forward (hook) → z (B,64) → grad → S (B,64)
        y → pick wrong c' → μ_{c'} (B,64)
        z' = z + ε_q * (S_norm ⊙ (μ_{c'} - z))
        z' → bridge → q_layer → classifier → logits → CE loss
    """
    model.eval()   # set eval for consistent BN/dropout during perturbation computation

    # ── Step 1: Extract z with gradient tracking ─────────────────────
    hook  = FeatureHook(model.gap)
    _     = model(x)                                          # full forward pass
    z_raw = hook.features.view(hook.features.size(0), -1)    # (B, 64, 1, 1) → (B, 64)
    hook.remove()

    z = z_raw.detach().requires_grad_(True)                   # enable grad on z

    # Recompute classifier path from z (bridge → q_layer → classifier)
    # This is the part of the forward pass AFTER the GAP hook point
    angles  = model.bridge(z)           # (B, 64) → (B, 16) quantum angles
    q_out   = model.q_layer(angles)     # (B, 16) → (B, 24) quantum measurements
    logits  = model.classifier(q_out)  # (B, 24) → (B, 10) class logits

    loss = F.cross_entropy(logits, y)   # scalar
    loss.backward()                     # compute ∂L/∂z → stored in z.grad

    # ── Step 2: Sensitivity vector ───────────────────────────────────
    S      = z.grad.detach()                              # (B, 64)
    S_norm = S / (S.norm(dim=1, keepdim=True) + 1e-8)    # ℓ2 normalise per sample

    # ── Step 3: Random wrong class per sample ────────────────────────
    z_detach = z.detach()
    target_classes = []
    for i in range(y.size(0)):
        wrong_classes = [c for c in range(centroids.size(0)) if c != y[i].item()]
        chosen = wrong_classes[torch.randint(0, len(wrong_classes), (1,)).item()]
        target_classes.append(chosen)
    wrong_idx  = torch.tensor(target_classes, device=device)
    mu_c_prime = centroids[wrong_idx]                     # (B, 64) wrong-class centroids

    # ── Step 4: Sensitivity-weighted perturbation ────────────────────
    delta   = mu_c_prime - z_detach                       # direction toward wrong centroid
    z_perturbed = z_detach + epsilon_q * (S_norm * delta) # (B, 64) perturbed features

    # ── Step 5: Forward from perturbed z → loss ──────────────────────
    model.train()   # restore train mode before computing loss for gradients
    angles_p  = model.bridge(z_perturbed)
    q_out_p   = model.q_layer(angles_p)
    logits_p  = model.classifier(q_out_p)
    loss_qni  = F.cross_entropy(logits_p, y)
    return loss_qni


# ─────────────────────────────────────────────
# FGSM ATTACK
# ─────────────────────────────────────────────
# Single-step pixel-space attack: x_adv = x + ε * sign(∇_x L)
# Fast but weak. Acts as a lightweight pixel-space regulariser.
# ε = 0.03 (paper Table 3)

def fgsm_attack(model, images, labels, eps=FGSM_EPS):
    """
    Generates FGSM adversarial images.
    Data flow: images → model → loss → ∂L/∂images → sign → ε * sign → add → clamp
    Returns: adversarial images (B,1,32,32) clamped to [-1,1]
    """
    model.eval()
    images_adv = images.clone().detach().requires_grad_(True)

    logits = model(images_adv)
    loss   = F.cross_entropy(logits, labels)
    model.zero_grad()
    loss.backward()

    # One step in the direction that maximises loss
    images_adv = images_adv + eps * images_adv.grad.sign()
    images_adv = torch.clamp(images_adv, -1.0, 1.0)
    return images_adv.detach()


# ─────────────────────────────────────────────
# PGD ATTACK
# ─────────────────────────────────────────────
# Iterative FGSM with projection back to ε-ball.
# Stronger than FGSM; the gold standard adversarial attack.
# ε=0.10, α=0.01, k=7 (paper Table 3)

def pgd_attack(model, images, labels,
               eps=PGD_EPS, alpha=PGD_ALPHA, iters=PGD_ITERS):
    """
    Generates PGD adversarial images.
    Data flow:
      Start: images + uniform noise in [-ε, ε]
      Each iter: → model → loss → sign(grad) → step α → project to ε-ball → clamp
    Returns: adversarial images (B,1,32,32)
    """
    model.eval()
    images_orig = images.clone().detach()

    # Random start inside ε-ball
    images_adv  = images_orig + torch.empty_like(images_orig).uniform_(-eps, eps)
    images_adv  = torch.clamp(images_adv, -1.0, 1.0).detach()

    for _ in range(iters):
        images_adv.requires_grad_(True)
        logits = model(images_adv)
        loss   = F.cross_entropy(logits, labels)
        model.zero_grad()
        loss.backward()

        # Step toward maximum loss
        step       = alpha * images_adv.grad.sign()
        images_adv = images_adv.detach() + step

        # Project back to ε-ball around original
        delta      = torch.clamp(images_adv - images_orig, -eps, eps)
        images_adv = torch.clamp(images_orig + delta, -1.0, 1.0).detach()

    return images_adv


# ══════════════════════════════════════════════════════════════════════════════
# ADVERSARIAL TRAINING EPOCH
# ══════════════════════════════════════════════════════════════════════════════

def train_adversarial_epoch(model, train_loader, optimizer, centroids, device):
    """
    One epoch of QSLP adversarial training.

    For each batch, computes 5 losses and combines them:
      loss = 0.50 * CE(clean)
           + 0.15 * CE(QNI-CCP latent perturbation)
           + 0.10 * CE(FGSM adversarial image)
           + 0.15 * CE(PGD adversarial image)
           + 0.10 * MSE(clean features, class centroid)

    Returns: (avg_combined_loss, clean_accuracy)
    Clean accuracy is tracked as proxy — val accuracy on clean data is the real metric.
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in tqdm(train_loader, desc="Adv Training", leave=False):
        x, y = x.to(device), y.to(device)

        # ── Loss 1: Clean ─────────────────────────────────────────
        # Normal forward pass on unperturbed images.
        # Weight 0.50 ensures the model doesn't sacrifice clean accuracy.
        model.train()
        logits_clean = model(x)                          # (B, 10)
        loss_clean   = F.cross_entropy(logits_clean, y)  # scalar

        # ── Loss 2: QNI-CCP (latent-space) ───────────────────────
        # Perturb the 64-dim GAP features toward the wrong class centroid,
        # weighted by gradient sensitivity. Forces latent robustness.
        # Weight 0.15: meaningful defence at the quantum interface.
        loss_qni = qni_ccp_loss(model, x, y, centroids, EPSILON_Q)

        # ── Loss 3: FGSM (pixel-space) ────────────────────────────
        # Single-step pixel attack. Quick adversarial signal.
        # Weight 0.10: lighter than PGD since FGSM is a weaker attacker.
        model.eval()
        x_fgsm      = fgsm_attack(model, x, y, eps=FGSM_EPS)
        model.train()
        logits_fgsm = model(x_fgsm)
        loss_fgsm   = F.cross_entropy(logits_fgsm, y)

        # ── Loss 4: PGD (pixel-space) ─────────────────────────────
        # 7-step iterative attack. Much stronger than FGSM.
        # Weight 0.15: strongest attack gets same weight as QNI-CCP.
        model.eval()
        x_pgd      = pgd_attack(model, x, y, eps=PGD_EPS, alpha=PGD_ALPHA, iters=PGD_ITERS)
        model.train()
        logits_pgd = model(x_pgd)
        loss_pgd   = F.cross_entropy(logits_pgd, y)

        # ── Loss 5: Centroid regularisation ───────────────────────
        # Pulls clean backbone features toward the correct class centroid.
        # Reduces intra-class variance → more compact, separable clusters.
        # Weight 0.10: auxiliary regulariser, not adversarial.
        z_clean      = get_gap_features(model, x)        # (B, 64) no-grad features
        centroid_reg = ((z_clean - centroids[y]) ** 2).mean()  # MSE to correct centroid

        # ── Combined QSLP loss (Algorithm 1) ──────────────────────
        loss = (W_CLEAN    * loss_clean   +   # 0.50
                W_QNI      * loss_qni     +   # 0.15
                W_CENTROID * centroid_reg)     # 0.10  → total = 1.00

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits_clean.argmax(1) == y).sum().item()
        total      += y.size(0)

    return total_loss / len(train_loader), correct / total


# ─────────────────────────────────────────────
# EVALUATION FUNCTIONS
# ─────────────────────────────────────────────
def evaluate_clean(model, dataloader, device):
    """Standard clean evaluation. Returns (avg_loss, accuracy)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y       = x.to(device), y.to(device)
            logits     = model(x)
            total_loss += F.cross_entropy(logits, y).item()
            correct    += (logits.argmax(1) == y).sum().item()
            total      += y.size(0)
    return total_loss / len(dataloader), correct / total


def evaluate_adversarial(model, dataloader, device):
    """
    Evaluates model robustness under FGSM and PGD attacks.
    Called once at the end to report final adversarial accuracy.

    Data flow per batch:
      x → fgsm_attack → adversarial images → model → accuracy
      x → pgd_attack  → adversarial images → model → accuracy
    Returns: (fgsm_accuracy, pgd_accuracy)
    """
    model.eval()
    fgsm_correct, pgd_correct, total = 0, 0, 0

    for x, y in tqdm(dataloader, desc="Adversarial Eval", leave=False):
        x, y = x.to(device), y.to(device)

        # FGSM robustness
        x_fgsm = fgsm_attack(model, x, y)
        model.eval()
        with torch.no_grad():
            fgsm_correct += (model(x_fgsm).argmax(1) == y).sum().item()

        # PGD robustness
        x_pgd = pgd_attack(model, x, y)
        with torch.no_grad():
            pgd_correct += (model(x_pgd).argmax(1) == y).sum().item()

        total += y.size(0)

    return fgsm_correct / total, pgd_correct / total


# ══════════════════════════════════════════════════════════════════════════════
# MAIN SCRIPT
# ══════════════════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────
# STEP 1: Build model and load pretrained weights
# ─────────────────────────────────────────────
model = HybridResNet(
    n_qubits    = n_qubits,
    q_out_dim   = q_out_dim,
    num_classes = num_classes,
    dropout     = 0.35
).to(device)

# Load your pretrained clean checkpoint
# Replace 'hybrid_resnet_NOGAN.pth' with your actual checkpoint filename
CHECKPOINT_PATH = 'hybrid_resnet_NOGAN.pth'
try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint from {CHECKPOINT_PATH}")
    print(f"Starting from Val Acc: {checkpoint.get('val_acc', 'unknown'):.4f}")
except FileNotFoundError:
    print(f"Checkpoint '{CHECKPOINT_PATH}' not found — training from scratch.")
    print("Tip: train clean first, then adversarially fine-tune for best results.")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")


# ─────────────────────────────────────────────
# STEP 2: Optimizer and scheduler
# ─────────────────────────────────────────────
# Standard AdamW (no SAM) for adversarial fine-tuning.
# Adversarial training itself provides strong implicit regularisation.
# Lower LR (0.0002 vs 0.0005) to avoid degrading pretrained clean accuracy.
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)


# ─────────────────────────────────────────────
# STEP 3: Compute initial class centroids
# ─────────────────────────────────────────────
print("\nComputing initial class centroids over training set...")
centroids = compute_class_centroids(model, train_loader, device, num_classes)
# centroids: (10, 64) — mean GAP features per class
print(f"Centroids computed. Shape: {centroids.shape}")


# ─────────────────────────────────────────────
# STEP 4: Adversarial training loop
# ─────────────────────────────────────────────
best_val_acc               = 0.0
early_stopping_patience    = 12     # slightly shorter patience for fine-tuning
epochs_without_improvement = 0
train_losses, val_losses   = [], []
train_accs,   val_accs     = [], []

print(f"\nStarting QSLP Adversarial Training for {num_epochs} epochs")
print(f"Attacks: FGSM(ε={FGSM_EPS}) | PGD(ε={PGD_EPS}, α={PGD_ALPHA}, k={PGD_ITERS})")
print(f"Weights: clean={W_CLEAN} | QNI={W_QNI} | FGSM={W_FGSM} | PGD={W_PGD} | centroid={W_CENTROID}")
print("=" * 70)

for epoch in range(1, num_epochs + 1):

    # Recompute centroids every 5 epochs (Algorithm 1 lines 3-5)
    # As the model updates, feature distributions shift — stale centroids
    # would give QNI-CCP inaccurate perturbation targets
    if epoch % 5 == 0:
        print(f"  🔄 Recomputing centroids at epoch {epoch}...")
        centroids = compute_class_centroids(model, train_loader, device, num_classes)

    train_loss, train_acc = train_adversarial_epoch(
        model, train_loader, optimizer, centroids, device
    )
    val_loss, val_acc = evaluate_clean(model, val_loader, device)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch [{epoch:02d}/{num_epochs}] | LR: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc (clean): {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f}  | Val   Acc (clean): {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc               = val_acc
        epochs_without_improvement = 0
        torch.save({
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc':              val_acc,
            'val_loss':             val_loss,
            'config': {
                'n_qubits':    n_qubits,
                'q_depth':     q_depth,
                'q_out_dim':   q_out_dim,
                'num_classes': num_classes,
                'adv_trained': True,
                'fgsm_eps':    FGSM_EPS,
                'pgd_eps':     PGD_EPS,
            }
        }, "hybrid_resnet_v3_adversarial.pth")
        print(f"  💾 Best model saved (Val Acc: {best_val_acc:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  🕒 No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= early_stopping_patience:
        print(f"\n⏹️  Early stopping at epoch {epoch}.")
        break

    print("-" * 70)

print(f"\n✅ Adversarial training complete. Best Val Acc: {best_val_acc:.4f}")


# ─────────────────────────────────────────────
# STEP 5: Final adversarial robustness evaluation
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("FINAL ADVERSARIAL ROBUSTNESS REPORT")
print("=" * 70)

checkpoint = torch.load("hybrid_resnet_v3_adversarial.pth", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

_, clean_acc      = evaluate_clean(model, test_loader, device)
fgsm_acc, pgd_acc = evaluate_adversarial(model, test_loader, device)

print(f"\n  Clean Test Accuracy : {clean_acc:.4f}  ({clean_acc*100:.1f}%)")
print(f"  FGSM  Test Accuracy : {fgsm_acc:.4f}  ({fgsm_acc*100:.1f}%)  ε={FGSM_EPS}")
print(f"  PGD   Test Accuracy : {pgd_acc:.4f}  ({pgd_acc*100:.1f}%)  ε={PGD_EPS}, k={PGD_ITERS}")
print(f"\n  FGSM robustness drop: {(clean_acc - fgsm_acc)*100:.1f}%")
print(f"  PGD  robustness drop: {(clean_acc - pgd_acc)*100:.1f}%")
print("\n  Compare these numbers against the clean baseline to measure QSLP benefit.")